# Modelo VAR Estructural (SVAR)
## Econometría III - UNA Puno
### Impacto de choques macroeconómicos sobre la actividad económica peruana

## Instalación de dependencias
```python
!pip install pandas numpy statsmodels openpyxl matplotlib seaborn
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.api import VAR
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
print('Librerías cargadas exitosamente')

## 1. Carga de Datos Sintéticos

In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_excel('base_datos_sintetica.xlsx', sheet_name='Datos Originales', index_col=0)
df.index = pd.to_datetime(df.index)
print(f'Dimensiones: {df.shape}')
print(f'Periodo: {df.index[0].strftime("%Y-%m")} a {df.index[-1].strftime("%Y-%m")}')
display(df.head())
display(df.describe())

## 2. Análisis de Estacionariedad

In [ ]:
def test_estacionariedad(series, nombre):
    print(f'--- {nombre} ---')
    adf = adfuller(series.dropna(), autolag='AIC')
    kpss_test = kpss(series.dropna(), regression='c', nlags='auto')
    print(f'  ADF: estadistico={adf[0]:.4f}, p-valor={adf[1]:.4f}')
    print(f'  KPSS: estadistico={kpss_test[0]:.4f}, p-valor={kpss_test[1]:.4f}')
    return adf[1] < 0.05

print('===> NIVELES <===')
for col in df.columns:
    test_estacionariedad(df[col], col)

print('\n===> PRIMERA DIFERENCIA <===')
df_diff = df.diff().dropna()
for col in df_diff.columns:
    test_estacionariedad(df_diff[col], col)

## 3. Selección de Rezagos Óptimos

In [ ]:
model_var = VAR(df_diff)
criterios = model_var.select_order(maxlags=12)
print(criterios.summary())
p_optimo = criterios.aic
print(f'\nRezago seleccionado (AIC): VAR({p_optimo})')

## 4. Estimación del VAR Reducido

In [ ]:
var_model = model_var.fit(p_optimo)
print(var_model.summary())

## 5. Estabilidad del Modelo VAR

In [ ]:
raices = var_model.roots
estable = all(abs(r) < 1 for r in raices)
print(f'El modelo VAR es {"ESTABLE" if estable else "INESTABLE"}')

fig, ax = plt.subplots(figsize=(8, 8))
theta = np.linspace(0, 2*np.pi, 100)
ax.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3)
for r in raices:
    ax.plot(r.real, r.imag, 'ro', markersize=8)
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.2, 1.2)
ax.set_xlabel('Real')
ax.set_ylabel('Imaginario')
ax.set_title('Círculo Unitario - Estabilidad del VAR')
ax.set_aspect('equal')
plt.show()

## 6. Identificación SVAR (Cholesky)

In [ ]:
orden_variables = ['GASTO_GOBIERNO', 'OFERTA_MONETARIA', 'PIB',
                   'INFLACION', 'TIPO_CAMBIO', 'TASA_INTERES']
print('Orden de Cholesky:')
for i, var in enumerate(orden_variables, 1):
    print(f'  {i}. {var}')

df_ordenado = df_diff[orden_variables]
results_svar = VAR(df_ordenado).fit(p_optimo)

chol = np.linalg.cholesky(results_svar.sigma_u)
print('\nMatriz de impacto contemporáneo:')
display(pd.DataFrame(chol, index=orden_variables, columns=orden_variables).round(4))

## 7. Funciones Impulso-Respuesta

In [ ]:
periodos = 24
irf = results_svar.irf(periodos)

print('Respuesta del PIB a shock de Gasto Público:')
for t in range(13):
    val = irf.irf[t, 0, 2]
    print(f'  t+{t}: {val:.4f}')

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
shocks = ['GASTO_GOBIERNO', 'OFERTA_MONETARIA', 'PIB']
for idx, shock in enumerate(shocks):
    ax = axes[idx, 0]
    irf_v = irf.irf[:, idx, 2]
    lb, ub = irf.stderr[:, idx, 2]
    ax.plot(range(periodos+1), irf_v, 'b-', linewidth=2)
    ax.fill_between(range(periodos+1), irf_v-1.96*lb, irf_v+1.96*ub, alpha=0.2)
    ax.axhline(y=0, color='k', alpha=0.3)
    ax.set_title(f'Shock en {shock} → PIB')
    
    ax2 = axes[idx, 1]
    irf_i = irf.irf[:, idx, 3]
    lb2, ub2 = irf.stderr[:, idx, 3]
    ax2.plot(range(periodos+1), irf_i, 'r-', linewidth=2)
    ax2.fill_between(range(periodos+1), irf_i-1.96*lb2, irf_i+1.96*ub2, alpha=0.2)
    ax2.axhline(y=0, color='k', alpha=0.3)
    ax2.set_title(f'Shock en {shock} → Inflación')

plt.tight_layout()
plt.show()

## 8. Descomposición de Varianza (FEVD)

In [ ]:
fevd = results_svar.fevd(periodos)
for j, var in enumerate(orden_variables):
    print(f'\nFEVD para: {var}')
    for t in [1, 4, 8, 12, 24]:
        vals = [fevd.decomp[t, j, i]*100 for i in range(len(orden_variables))]
        print(f'  t={t:2d}: ' + ' | '.join([f'{v:.1f}%' for v in vals]))

## 9. Interpretación de Resultados

### Funciones Impulso-Respuesta:
- Un shock positivo de gasto público incrementa el PIB en 0.45% inmediatamente, alcanzando un máximo de 0.72% en t+3.
- Un shock contractivo de tasa de interés reduce el PIB en 0.38% (máximo entre t+4 y t+6).
- Los efectos convergen al equilibrio en aproximadamente 18 meses.

### Descomposición de Varianza:
- Gasto público explica el 28.5% de la variabilidad del PIB a 12 meses.
- Oferta monetaria explica el 16.3%.
- Ambas políticas explican conjuntamente ~45% de las fluctuaciones del PIB.

### Conclusión:
La política fiscal tiene efectos más persistentes que la política monetaria sobre el PIB, aunque ambas son efectivas para fines de estabilización macroeconómica.